### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [ ]:
# Agents + Memory + Checkpointing + Automatic Summarization
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
# Saves conversation state in memory so it can be resumed using a thread ID.
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model="nvidia:nvidia/nemotron-3.5-super-vl-120b-a12b",
    checkpointer=InMemorySaver(), 
    # very imp --> save in memory(create a checkpoint) so that agent persists its state btw seperate invocations
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b", #less cost wale rakha karo for summarization
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [ ]:
# Run with thread id --> to uniquely identify a user, so that  
# when langchain retrieves the prev conversations it knows kiski uthaani
config={"configurable":{"thread_id":"test-1"}}

In [4]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='7e0ed539-4499-4434-af82-d62cc06e0d1d'), AIMessage(content='2 + 2 = 4', additional_kwargs={'reasoning_content': 'The user is asking a simple arithmetic question: "What is 2+2?"\nThe answer is 4.\nI should provide a direct, clear, and concise answer.\n\nPlan:\n1. State the answer clearly.', '_reasoning_api_fields': ['reasoning_content']}, response_metadata={'content': '2 + 2 = 4', 'refusal': None, 'role': 'assistant', 'reasoning_content': 'The user is asking a simple arithmetic question: "What is 2+2?"\nThe answer is 4.\nI should provide a direct, clear, and concise answer.\n\nPlan:\n1. State the answer clearly.', 'token_usage': {'prompt_tokens': 23, 'completion_tokens': 53, 'total_tokens': 76, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 44}}, 'finish_reason': 'stop', 'model_name': 'nvidia/nemotron-3.5-super-vl-120b-a12b'}, id='

## token Size

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
 
@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="nvidia:nvidia/nemotron-3.5-super-vl-120b-a12b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.6-flash",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [6]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~99 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='9d9501f1-ee35-4836-aaff-f3548608b587'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to find hotels in Paris.\nI have a tool named `search_hotels` that takes a `city` parameter.\nI should call `search_hotels` with `city` set to "Paris".\n\nTool: `search_hotels`\nParameter: `city` = "Paris"\n\nLet\'s generate the tool call.', '_reasoning_api_fields': ['reasoning_content'], 'tool_calls': [{'id': 'call-5d041a9b-d59d-405f-a262-a887064b6168', 'type': 'function', 'function': {'name': 'search_hotels', 'arguments': '{"city":"Paris"}'}}]}, response_metadata={'content': None, 'refusal': None, 'tool_calls': [{'id': 'call-5d041a9b-d59d-405f-a262-a887064b6168', 'type': 'function', 'function': {'name': 'search_hotels', 'arguments': '{"city":"Paris"}'}}], 'role': 'assistant', 'reasoning_content': 'The user wants to find hotels in Paris.\nI have a

### Fraction

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="nvidia:nvidia/nemotron-3.5-super-vl-120b-a12b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.6-flash",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 262144  # nemotrom-3.5 context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~47 tokens (0.0179%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='2dc5d5f2-1fa6-4102-b166-8ff2b2f54526'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking for hotels in Paris. I have a tool `search_hotels` that takes a `city` parameter. I should call this tool with `city` set to "Paris".\n\nPlan:\n1. Call `search_hotels` with `city="Paris"`.\n2. Present the results to the user.\n\nConstraint checklist & Confidence score:\n1. Search hotels? Yes.\n2. Tool available? Yes.\nConfidence Score: 5/5\n\nMental Sandbox:\n- If I call the tool, I will get hotel data.\n- I should format the response nicely.\n- No extra steps needed.\n\nKey learnings from sandbox:\n- Direct tool usage is appropriate.\n\nStrategizing complete. I will now generate the tool call.', '_reasoning_api_fields': ['reasoning_content'], 'tool_calls': [{'id': 'call-8c997ba6-e1cc-458a-aec8-a657a1dd11ec', 'type': 'function', 'function': {'

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [12]:
agent=create_agent(
    model="nvidia:nvidia/nemotron-3.5-super-vl-120b-a12b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [13]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [14]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c9485299-b5d6-4764-bd15-43a4a0b235aa'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email.\nI have the `send_email_tool` available.\nParameters required:\n- recipient: "john@test.com"\n- subject: "Hello"\n- body: "How are you?"\n\nI will call the tool with these parameters.', '_reasoning_api_fields': ['reasoning_content'], 'tool_calls': [{'id': 'call-ade3f6ef-71df-477a-a729-a236d316172f', 'type': 'function', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient":"john@test.com","subject":"Hello","body":"How are you?"}'}}]}, response_metadata={'content': None, 'refusal': None, 'tool_calls': [{'id': 'call-ade3f6ef-71df-477a-a729-a236d316172f', 'type': 'function', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient":"john@test.com","subject":"Hello","body":"How

In [ ]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
# Command is an object used to control 
# what happens next in a LangGraph workflow.
    # Resume a paused workflow.
    #Send information back into a workflow.
    #Control which node executes next(using other Command options).
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
# when approved langgraph allows send_email_tool to execute
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been successfully sent to john@test.com with the subject 'Hello'.


In [16]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c9485299-b5d6-4764-bd15-43a4a0b235aa'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email.\nI have the `send_email_tool` available.\nParameters required:\n- recipient: "john@test.com"\n- subject: "Hello"\n- body: "How are you?"\n\nI will call the tool with these parameters.', '_reasoning_api_fields': ['reasoning_content'], 'tool_calls': [{'id': 'call-ade3f6ef-71df-477a-a729-a236d316172f', 'type': 'function', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient":"john@test.com","subject":"Hello","body":"How are you?"}'}}]}, response_metadata={'content': None, 'refusal': None, 'tool_calls': [{'id': 'call-ade3f6ef-71df-477a-a729-a236d316172f', 'type': 'function', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient":"john@test.com","subject":"Hello","body":"How

### Reject

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="nvidia:nvidia/nemotron-3.5-super-vl-120b-a12b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [18]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [19]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I attempted to send the email but the tool call was rejected. Would you like me to try sending the email again, or is there anything else you'd like to do?


In [20]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='b898f04f-ea10-4978-9d84-ea45850db131'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email with:\n- recipient: john@test.com\n- subject: 'Hello'\n- body: 'How are you?'\n\nI should call the send_email_tool function with these parameters.", '_reasoning_api_fields': ['reasoning_content'], 'tool_calls': [{'id': 'call-4d3ee322-2974-423a-a951-8dc5c495d9a6', 'type': 'function', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient":"john@test.com","subject":"Hello","body":"How are you?"}'}}]}, response_metadata={'content': None, 'refusal': None, 'tool_calls': [{'id': 'call-4d3ee322-2974-423a-a951-8dc5c495d9a6', 'type': 'function', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient":"john@test.com","subject":"Hello","body":"How are you?"}'}}], 'role': 'assistant', 

### Editing

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="nvidia:nvidia/nemotron-3.5-super-vl-120b-a12b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [22]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [23]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='2dfd4cbf-aa1d-4c8d-9013-865e2f8a1d89'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email with specific details:\n- Recipient: wrong@email.com\n- Subject: Test\n- Body: Hello\n\nI should use the `send_email_tool` function with these parameters.\nLet\'s check the function definition:\n`send_email_tool(recipient: string, subject: string, body: string)`\n\nParameters:\n- recipient: "wrong@email.com"\n- subject: "Test"\n- body: "Hello"\n\nAll required parameters are provided. I will proceed with the function call.', '_reasoning_api_fields': ['reasoning_content'], 'tool_calls': [{'id': 'call-e8b64a07-e0fa-4781-9b14-f16e7b529430', 'type': 'function', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient":"wrong@email.com","subject":"Test","body":"Hello"}'}}]}, response_metadata={'con

In [24]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: The email was sent, but with modified details before execution:

- **Recipient:** correct@email.com (instead of wrong@email.com)
- **Subject:** Corrected Subject (instead of Test)
- **Body:** This was edited by human before sending (instead of Hello)

Status: Email sent successfully to correct@email.com.


In [25]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='2dfd4cbf-aa1d-4c8d-9013-865e2f8a1d89'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email with specific details:\n- Recipient: wrong@email.com\n- Subject: Test\n- Body: Hello\n\nI should use the `send_email_tool` function with these parameters.\nLet\'s check the function definition:\n`send_email_tool(recipient: string, subject: string, body: string)`\n\nParameters:\n- recipient: "wrong@email.com"\n- subject: "Test"\n- body: "Hello"\n\nAll required parameters are provided. I will proceed with the function call.', '_reasoning_api_fields': ['reasoning_content'], 'tool_calls': [{'id': 'call-e8b64a07-e0fa-4781-9b14-f16e7b529430', 'type': 'function', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient":"wrong@email.com","subject":"Test","body":"Hello"}'}}]}, response_metadata={'con